# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nimrafarooq2006-coder/ml-internship-work/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

Main queue (option B): pages with at least 100 impressions in 90 days,
ranked by a logistic regression score. Every page is scored by a
model that never saw its client (5-fold grouped by client, 32
clients). Of the top 100, 78% are marked declining, against a
reference of 59.8% among pages with 100+ impressions (top 50: 68%,
top 500: 76.2%). AUC overall is only 0.57, so the ranking is rough.

Reason code to action (my replacement for archetype to action, because
my lane is scoring, not clustering):
- stale_visible_page (90+ days since update, 100+ impressions): Review
  and refresh. 8,119 pages.
- low_ctr_visible_page (500+ impressions, position 1-20, CTR under
  0.5): Review title and snippet. 6,225 pages.
- thin_visible_page (under 1,200 words, 250+ impressions): Review depth
  and expand. 42 pages.
- model_risk_only (no rule fired, model score only): Monitor. 15,614
  pages.

Decay and refresh insight: pages last updated 91-180 days ago were
marked declining most often (61.1%, n=9,171), against 51.1% for pages
updated in the last 30 days (n=20,480). The 31-90 and 181+ groups are
too small to trust (n=175 and 174). This is an observed association.
It does not show that a refresh helps.

Two orders I tried and rejected: model score alone (A) filled the top
with pages almost nobody visits (median 54 impressions). Weighting by
visibility (C) fell below random picking (0.44 at top 50 against 0.542)
and put 97 of the top 100 on two clients.

In [10]:
import os, json, subprocess
import pandas as pd, numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

path = "data/raw/content_refresh_anonymized.csv"
if not os.path.exists(path):
    if not os.path.isdir("ml-internship-work"):
        subprocess.run(["git", "clone", "--depth", "1",
            "https://github.com/nimrafarooq2006-coder/ml-internship-work"], check=True)
    path = "ml-internship-work/" + path

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

df = pd.read_csv(path)
df["declining"] = df["trend_direction"].str.lower().eq("down").astype(int)
features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["declining"].values
groups = df["client_id"]

# Every page is scored by a model that never saw its client (5 grouped folds)
oof = np.zeros(len(df))
for tr, te in GroupKFold(n_splits=5).split(X, y, groups):
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X.iloc[tr], y[tr])
    oof[te] = m.predict_proba(X.iloc[te])[:, 1]
df["model_score"] = (oof * 100).round(1)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

metrics = {
    "pages": int(len(df)), "clients": int(groups.nunique()),
    "validation": "5-fold grouped by client (no client in both train and test)",
    "random_picking_rate": round(float(y.mean()), 3),
    "auc": round(float(roc_auc_score(y, oof)), 3),
    "precision_at_50": round(precision_at_k(oof, y, 50), 3),
    "precision_at_100": round(precision_at_k(oof, y, 100), 3),
    "precision_at_500": round(precision_at_k(oof, y, 500), 3),
}

# Reason codes (first match wins) and action labels
imp, pos, ctr, wc = df["impressions_90d"], df["avg_position"], df["ctr"], df["word_count"]
conds = [
    (df["days_since_last_update"] >= 90) & (imp >= 100),
    (imp >= 500) & (pos > 0) & (pos <= 20) & (ctr < 0.5),
    (wc > 0) & (wc < 1200) & (imp >= 250),
]
codes = ["stale_visible_page", "low_ctr_visible_page", "thin_visible_page"]
actions = ["Review and refresh", "Review title and snippet", "Review depth and expand"]
df["reason_code"] = np.select(conds, codes, default="model_risk_only")
df["action"] = np.select(conds, actions, default="Monitor")

cut = df["model_score"].quantile(0.80)
df["confidence"] = np.where((df["model_score"] >= cut) & (imp >= 500), "high",
                    np.where(df["model_score"] >= cut, "medium", "low"))

queue = df.sort_values("model_score", ascending=False).reset_index(drop=True)
queue.insert(0, "rank", queue.index + 1)
cols = ["rank", "content_id", "client_id", "model_score", "reason_code", "action", "confidence",
        "impressions_90d", "days_since_last_update", "content_age_days", "avg_position", "ctr", "word_count"]
queue[cols].to_csv("work/outputs/ranked_action_queue.csv", index=False)
with open("work/outputs/playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

# Figure 1: decay by days since last update
b = pd.cut(df["days_since_last_update"], [-1, 30, 90, 180, 10**6],
           labels=["0-30", "31-90", "91-180", "181+"])
decay = df.groupby(b, observed=True)["declining"].agg(["size", "mean"])
plt.figure(figsize=(6, 3.5)); plt.bar(decay.index.astype(str), decay["mean"], color="#c2410c")
plt.axhline(y.mean(), color="gray", ls="--", label="all pages")
plt.xlabel("Days since last update"); plt.ylabel("Share marked declining"); plt.legend()
plt.title("Observed decline rate by days since update"); plt.tight_layout()
plt.savefig("work/figures/decay_by_update_age.png", dpi=150); plt.close()

print(json.dumps(metrics, indent=2))
print("\nDecline rate by days since update (n shown):")
print(decay.round(3))
print("\nReason codes in the whole queue:")
print(queue["reason_code"].value_counts())
print("\nTop 10 of the queue:")
queue[cols].head(10)


{
  "pages": 30000,
  "clients": 32,
  "validation": "5-fold grouped by client (no client in both train and test)",
  "random_picking_rate": 0.542,
  "auc": 0.57,
  "precision_at_50": 0.64,
  "precision_at_100": 0.66,
  "precision_at_500": 0.686
}

Decline rate by days since update (n shown):
                         size   mean
days_since_last_update              
0-30                    20480  0.511
31-90                     175  0.589
91-180                   9171  0.611
181+                      174  0.471

Reason codes in the whole queue:
reason_code
model_risk_only         15614
stale_visible_page       8119
low_ctr_visible_page     6225
thin_visible_page          42
Name: count, dtype: int64

Top 10 of the queue:


,rank,content_id,client_id,model_score,reason_code,action,confidence,impressions_90d,days_since_last_update,content_age_days,avg_position,ctr,word_count
0,1,content_1b4ec72dafd4,client_4ec9599fc2,85.5,model_risk_only,Monitor,medium,2,372,372,7.0,0.0,NaN
1,2,content_55a5b1c46474,client_4ec9599fc2,85.5,model_risk_only,Monitor,medium,35,373,374,7.5,0.0,NaN
2,3,content_f6fdf87348f6,client_4ec9599fc2,85.0,model_risk_only,Monitor,medium,2,373,373,32.5,0.0,NaN
3,4,content_8d56efff1e71,client_4ec9599fc2,84.9,model_risk_only,Monitor,medium,1,372,372,35.0,0.0,NaN
4,5,content_06e19c6486b0,client_4ec9599fc2,84.3,model_risk_only,Monitor,medium,10,334,334,5.0,0.0,1300.0
5,6,content_e2b702f4f92b,client_4ec9599fc2,84.2,model_risk_only,Monitor,medium,30,334,334,9.3,0.0,1246.0
6,7,content_6476d1d8c050,client_19581e27de,83.2,stale_visible_page,Review and refresh,medium,304,313,313,67.8,0.0,NaN
7,8,content_7a888d3d99c8,client_19581e27de,83.2,model_risk_only,Monitor,medium,95,313,313,67.6,0.0,NaN
8,9,content_07ce98c6085a,client_4ec9599fc2,82.9,model_risk_only,Monitor,medium,85,304,305,5.3,0.0,1442.0
9,10,content_ab27c30d81f4,client_4ec9599fc2,82.6,stale_visible_page,Review and refresh,medium,103,304,305,8.9,0.0,1094.0


In [11]:
# Cost/value fix: actionable pages first, ranked by model score x visibility
df["priority"] = (df["model_score"] / 100) * np.log10(df["impressions_90d"].clip(lower=1) + 1)
df["actionable"] = df["reason_code"] != "model_risk_only"
df["queue_group"] = np.where(df["actionable"], "actionable", "monitor_only")

new_q = df.sort_values(["actionable", "priority"], ascending=[False, False]).reset_index(drop=True)
new_q.insert(0, "rank", new_q.index + 1)

def p_at(q, k): return round(float(q["declining"].head(k).mean()), 3)
old_q = queue
print("Precision@K, old order (model score only) vs new order (actionable first, score x visibility):")
for k in (50, 100, 500):
    print(f"  top {k}: old {p_at(old_q, k)} | new {p_at(new_q, k)} | random picking {metrics['random_picking_rate']}")

print("\nMedian impressions in top 50: old", old_q["impressions_90d"].head(50).median(),
      "| new", new_q["impressions_90d"].head(50).median())
print("\nClients in the top 100, new order:")
print(new_q["client_id"].head(100).value_counts().head(5))
print("\nNew top 10 reason codes:", new_q["reason_code"].head(10).tolist())

cols2 = cols + ["priority", "queue_group"]
new_q[["rank"] + [c for c in cols2 if c != "rank"]].to_csv("work/outputs/ranked_action_queue.csv", index=False)
metrics.update({"queue_order": "actionable reason codes first, then model score x log10(impressions)",
                "queue_precision_at_50": p_at(new_q, 50), "queue_precision_at_100": p_at(new_q, 100),
                "queue_precision_at_500": p_at(new_q, 500)})
with open("work/outputs/playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
new_q[["rank"] + [c for c in cols2 if c != "rank"]].head(10)

Precision@K, old order (model score only) vs new order (actionable first, score x visibility):
  top 50: old 0.64 | new 0.44 | random picking 0.542
  top 100: old 0.66 | new 0.42 | random picking 0.542
  top 500: old 0.686 | new 0.556 | random picking 0.542

Median impressions in top 50: old 54.0 | new 81664.5

Clients in the top 100, new order:
client_id
client_6208ef0f77    62
client_19581e27de    35
client_7f2253d7e2     3
Name: count, dtype: int64

New top 10 reason codes: ['stale_visible_page', 'stale_visible_page', 'stale_visible_page', 'stale_visible_page', 'stale_visible_page', 'stale_visible_page', 'stale_visible_page', 'stale_visible_page', 'stale_visible_page', 'stale_visible_page']


,rank,content_id,client_id,model_score,reason_code,action,confidence,impressions_90d,days_since_last_update,content_age_days,avg_position,ctr,word_count,priority,queue_group
0,1,content_a69b8a33d165,client_6208ef0f77,70.3,stale_visible_page,Review and refresh,high,64865,104,148,10.1,0.14,7537.0,3.382848,actionable
1,2,content_4a6607efcb46,client_6208ef0f77,66.1,stale_visible_page,Review and refresh,high,128068,104,148,2.2,0.01,4939.0,3.376020,actionable
2,3,content_c55195ea8c1f,client_6208ef0f77,68.9,stale_visible_page,Review and refresh,high,72715,104,144,29.1,0.11,7136.0,3.349663,actionable
3,4,content_6d7c9e5047f8,client_6208ef0f77,68.9,stale_visible_page,Review and refresh,high,72309,104,144,39.0,0.06,7582.0,3.347988,actionable
4,5,content_1a73ab830798,client_6208ef0f77,67.1,stale_visible_page,Review and refresh,high,95123,104,144,25.9,0.09,5858.0,3.340433,actionable
5,6,content_c22c1dd7122d,client_6208ef0f77,68.3,stale_visible_page,Review and refresh,high,76796,104,153,14.9,0.17,6394.0,3.336690,actionable
6,7,content_6a5b8ccbd700,client_6208ef0f77,64.5,stale_visible_page,Review and refresh,high,148534,104,153,18.3,0.60,5853.0,3.335830,actionable
7,8,content_475a9c3da500,client_6208ef0f77,68.0,stale_visible_page,Review and refresh,high,77651,104,139,24.1,0.31,6055.0,3.325304,actionable
8,9,content_b28d1efd668f,client_6208ef0f77,60.7,stale_visible_page,Review and refresh,low,286608,104,153,26.2,0.06,6901.0,3.312575,actionable
9,10,content_365128b9f3b7,client_6208ef0f77,67.5,stale_visible_page,Review and refresh,high,80377,104,144,35.0,0.13,6299.0,3.310968,actionable


In [12]:
# Option B: model score order, but only for pages with enough visibility
vis = df[df["impressions_90d"] >= 100]
qB = vis.sort_values("model_score", ascending=False).reset_index(drop=True)
qB.insert(0, "rank", qB.index + 1)
base_vis = round(float(vis["declining"].mean()), 3)

print("Pages with 100+ impressions:", len(vis), "| their decline rate (fair random reference):", base_vis)
print("\nPrecision@K for all three orders:")
for k in (50, 100, 500):
    print(f"  top {k}: A model only {p_at(old_q, k)} | B model, visible pages only {p_at(qB, k)} | C score x visibility {p_at(new_q, k)}")
print("\nMedian impressions in top 50: A", old_q["impressions_90d"].head(50).median(),
      "| B", qB["impressions_90d"].head(50).median(), "| C", new_q["impressions_90d"].head(50).median())
print("\nClients in the top 100 of B:")
print(qB["client_id"].head(100).value_counts().head(5))
print("\nReason codes in the top 100 of B:")
print(qB["reason_code"].head(100).value_counts())

# Export: B is the main queue, C is kept as an alternative view
qB[cols].to_csv("work/outputs/ranked_action_queue.csv", index=False)
new_q[["rank"] + [c for c in cols2 if c != "rank"]].to_csv("work/outputs/ranked_queue_visibility_weighted.csv", index=False)
metrics.update({
    "main_queue": "B: model score order, pages with 100+ impressions only",
    "visible_pages": int(len(vis)), "visible_decline_rate": base_vis,
    "B_precision_at_50": p_at(qB, 50), "B_precision_at_100": p_at(qB, 100), "B_precision_at_500": p_at(qB, 500),
    "A_precision_at_50": p_at(old_q, 50), "C_precision_at_50": p_at(new_q, 50)})
with open("work/outputs/playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
qB[cols].head(10)

Pages with 100+ impressions: 22006 | their decline rate (fair random reference): 0.598

Precision@K for all three orders:
  top 50: A model only 0.64 | B model, visible pages only 0.68 | C score x visibility 0.44
  top 100: A model only 0.66 | B model, visible pages only 0.78 | C score x visibility 0.42
  top 500: A model only 0.686 | B model, visible pages only 0.762 | C score x visibility 0.556

Median impressions in top 50: A 54.0 | B 1565.5 | C 81664.5

Clients in the top 100 of B:
client_id
client_6208ef0f77    70
client_19581e27de    11
client_7f2253d7e2    11
client_4ec9599fc2     3
client_9400f1b21c     3
Name: count, dtype: int64

Reason codes in the top 100 of B:
reason_code
stale_visible_page    99
model_risk_only        1
Name: count, dtype: int64


,rank,content_id,client_id,model_score,reason_code,action,confidence,impressions_90d,days_since_last_update,content_age_days,avg_position,ctr,word_count
0,1,content_6476d1d8c050,client_19581e27de,83.2,stale_visible_page,Review and refresh,medium,304,313,313,67.8,0.00,NaN
1,2,content_ab27c30d81f4,client_4ec9599fc2,82.6,stale_visible_page,Review and refresh,medium,103,304,305,8.9,0.00,1094.0
2,3,content_72496874f806,client_4ec9599fc2,82.5,stale_visible_page,Review and refresh,high,821,301,301,5.8,0.24,1504.0
3,4,content_d25a099b3726,client_19581e27de,82.4,stale_visible_page,Review and refresh,medium,202,305,313,64.5,0.00,NaN
4,5,content_4729b57ca036,client_4ec9599fc2,79.8,stale_visible_page,Review and refresh,medium,335,301,301,6.8,3.28,1397.0
5,6,content_02b0d6e30129,client_19581e27de,79.4,stale_visible_page,Review and refresh,medium,176,313,313,6.9,0.00,NaN
6,7,content_4f241bad48a3,client_6208ef0f77,79.1,stale_visible_page,Review and refresh,medium,285,236,236,19.1,0.00,6140.0
7,8,content_f488400fca67,client_9400f1b21c,78.8,stale_visible_page,Review and refresh,medium,155,305,306,5.7,0.00,1113.0
8,9,content_df1fa766cac2,client_9400f1b21c,78.4,stale_visible_page,Review and refresh,medium,206,304,305,4.8,0.49,1146.0
9,10,content_7f116ae1f6f5,client_9400f1b21c,78.3,stale_visible_page,Review and refresh,high,954,301,301,9.0,0.42,1335.0


## 2. Intended use and limits

Intended use: a rough review order for a human content reviewer who has
limited time. Cost/value: if a reviewer can check 100 pages, option B
gives about 78 pages that are marked declining, against about 60 from
picking visible pages at random. That saves some wasted reviews, and it
does not tell the reviewer what to change.

Limits: the label is a simple proxy (trend_direction is "down"), not a
future outcome, so the model shows association, not prediction. AUC is
0.57. The data is a 30,000-page starter slice. 70 of the top 100 come
from one client, so the queue may reflect that client's site more than a
general pattern. I chose option B after seeing results on the same
pages, so its numbers are directional. Almost the whole top 100 is
stale_visible_page, so the model mostly orders pages the rule already
flags.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

Human review rules: a person reads every page before any change. They
check the page's real traffic history for seasonality, consolidation or
a normal dip. They check whether a similar page took over its demand.
They record why they accepted or rejected each recommendation. Pages
from any one client should not fill more than a third of a review batch.

Do NOT automate: editing, deleting, merging or de-indexing any page;
telling a client a page "will recover"; sending recommendations
without human review; ranking or judging clients; using trend_direction
or trend_pct as inputs (they are the label).

No-go cases: do not use this queue for pages with fewer than 100
impressions, for new pages under 30 days, or for any decision about
people. No client names, domains, URLs or raw queries appear in any
public output.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers
Light, non-production plan. Each month, compute the share of the top
100 that turn out declining on fresh data.
- Retrain or rebuild if that share stays at or below the random
  reference (about 0.60) for two months in a row.
- Check the data if the median impressions of the visible pages moves
  by more than 30%, or if a column changes meaning.
- Re-check the queue if one client takes more than half of the top 100.
- Re-fit at least once a quarter, with the same grouped validation.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [16]:
for f in ["work/outputs/ranked_action_queue.csv",
          "work/outputs/ranked_queue_visibility_weighted.csv",
          "work/outputs/playbook_metrics.json",
          "work/figures/decay_by_update_age.png"]:
    print(f, "| exists:", os.path.exists(f))
print(open("work/outputs/playbook_metrics.json").read())


work/outputs/ranked_action_queue.csv | exists: True
work/outputs/ranked_queue_visibility_weighted.csv | exists: True
work/outputs/playbook_metrics.json | exists: True
work/figures/decay_by_update_age.png | exists: True
{
  "pages": 30000,
  "clients": 32,
  "validation": "5-fold grouped by client (no client in both train and test)",
  "random_picking_rate": 0.542,
  "auc": 0.57,
  "precision_at_50": 0.64,
  "precision_at_100": 0.66,
  "precision_at_500": 0.686,
  "queue_order": "actionable reason codes first, then model score x log10(impressions)",
  "queue_precision_at_50": 0.44,
  "queue_precision_at_100": 0.42,
  "queue_precision_at_500": 0.556,
  "main_queue": "B: model score order, pages with 100+ impressions only",
  "visible_pages": 22006,
  "visible_decline_rate": 0.598,
  "B_precision_at_50": 0.68,
  "B_precision_at_100": 0.78,
  "B_precision_at_500": 0.762,
  "A_precision_at_50": 0.64,
  "C_precision_at_50": 0.44
}


Exports for the paper: ranked_action_queue.csv (option B queue, stays
out of git, regenerated by this notebook), ranked_queue_visibility_weighted.csv
(the rejected option C, for the paper's honest account),
playbook_metrics.json (the receipts for my numbers) and
decay_by_update_age.png (the decay figure).

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.